# 04e · Ablations (C6)

**Purpose.** R1: T' at half and double the default. R2: steady state features (1 to 4) vs all 12. R3: 16 vs 32 vs 64 features per sensor.

**Inputs**
- Both `X_signals.npy` + `meta.csv`

**Outputs**
- R1 GWT (600, 3600) / (600, 14400); ES (144, 385) / (144, 1540)
- R2 GWT (600, 288) vs (600, 864); ES (144, 44) vs (144, 132)
- R3 GWT (600, 1152) / (600, 2304) / (600, 4608); ES (144, 176) / (144, 352) / (144, 704)
- `results/04e_ablations.csv`

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). Before running, upload
**`results.zip`** (results of notebooks 01-03) to `/content` with the Files panel on the left. The cell clones the
repo, downloads the processed data from the shared Drive folder, unpacks the results, and moves into this
notebook's folder so the rest runs unchanged. On a laptop it does nothing.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 2b. results of the earlier notebooks (01, 02, 03 and the 04 ones already run), from results.zip
    #     uploaded to /content (Files panel on the left). The zip holds a results/ folder (+ report/tables/).
    if os.path.exists("/content/results.zip"):
        shutil.unpack_archive("/content/results.zip", REPO)
    have = sorted(os.path.basename(f) for f in glob.glob(f"{REPO}/results/*.csv"))
    print("results:", have)
    main = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
            "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
    missing = [n for n in main if f"{n}.csv" not in have]
    assert not missing, f"upload results.zip to /content first (missing: {missing})"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/04_comparisons")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import load_baseline_level, load_processed, precomputed, run_cv, save_results
from src.features.encoder import encoder_builder
from src.features.handcrafted import STEADY_STATE, handcrafted_features
from src.features.raw import raw_features

NAME = "04e_ablations"
TABLES = ROOT / "report/tables"
TASKS_ABL = ["multi"]       # add "binary" if time allows (R3 ablations are the slow part)
T_PRIME = {"gwt": 100, "es": 70}

## C6: small ablations (the defaults come from the main results)
| Representation | Variants run here | Default (main results) |
|---|---|---|
| R1 | T' at half and double the default | T' = 100 (GWT), 70 (ES) |
| R2 | steady state features 1 to 4 only | all 12 |
| R3 | 16 and 64 features per sensor | 32 |

In [ ]:
parts = []
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)                       # (N, C, T)
    levels = load_baseline_level(dataset)                         # (N, C)
    t = T_PRIME[dataset]
    runs = [
        (f"R1_raw_T{t // 2}", raw_features(X, t // 2), precomputed),            # (N, C * T'/2)
        (f"R1_raw_T{t * 2}", raw_features(X, t * 2), precomputed),              # (N, C * 2T')
        ("R2_steady_state_1to4", handcrafted_features(X, levels, info["n_base"], info["exposure_end"],
                                                      subset=STEADY_STATE), precomputed),   # (N, C * 4)
        ("R3_encoder_16", X, encoder_builder(n_out=16)),                       # (N, C * 16)
        ("R3_encoder_64", X, encoder_builder(n_out=64)),                       # (N, C * 64)
    ]
    for task in TASKS_ABL:
        for rep, data, builder in runs:
            s, _ = run_cv(data, meta, dataset, task, rep, builder, verbose=False)
            parts.append(s)
            print(dataset, task, rep, round(s.groupby("head")["macro_f1"].mean().max(), 3))
scores = pd.concat(parts, ignore_index=True)
save_results(scores, None, NAME)

In [ ]:
main = pd.concat([pd.read_csv(ROOT / f"results/{n}.csv") for n in
                  ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted", "03a_gwt_encoder", "03b_es_encoder"]])
main = main[main["task"].isin(TASKS_ABL)].replace({"rep": {"R1_raw": "R1_raw (default)", "R2_handcrafted": "R2 all 12 (default)",
                                                           "R3_encoder": "R3_encoder_32 (default)"}})
allv = pd.concat([main, scores], ignore_index=True)
table = (allv.groupby(["dataset", "task", "rep", "head"])["macro_f1"].mean().round(3)
         .unstack("head")[["logreg", "svm", "rf", "mlp"]])
table.to_csv(TABLES / "C6_ablations.csv")
table

## Colab: save the outputs
Colab deletes everything when the session ends. This zips `results/` and `report/` into `outputs_<notebook>.zip`,
copies it to your Google Drive (`MyDrive/gas-repr-guide/`) and downloads it. On a laptop it does nothing.

In [ ]:
if IN_COLAB:
    out_zip = shutil.make_archive(f"/content/outputs_04e_ablations", "zip", REPO, "results")
    shutil.make_archive(f"/content/report_04e_ablations", "zip", REPO, "report")
    files_out = [out_zip, f"/content/report_04e_ablations.zip"]
    print(files_out)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide"
        os.makedirs(out_dir, exist_ok=True)
        for f in files_out:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in files_out:
        files.download(f)